## Детекция клеток

Ваша задача: обучить YOLO для детекции дрожжевых клеток и микроструктур (см. [07_object_detection.ipynb](../workshops/07_object_detection.ipynb)). Всё необходимое для запуска обучения вы можете взять из ноутбука с практикой, доделать нужно будет самую малость:
- реализовать расчёт Mean Average Precision для всего валидационного сета
- попробовать привести regression loss к виду, который используется в Yolo9000 и YoloV3
- подобрать лучшие размеры якорных рамок с помощью кластеризации

Основная цель: любыми средствами добиться $mAP > 0.6$ на валидации.

Используйте класс `torchmetrics.detection.MeanAveragePrecision` для расчёта $mAP$.

Нас будет интересовать именно значение `map` в словаре со всеми метриками - это mean average precision, усреднённый по всем отсечкам intersection over union в диапазоне $[0.5, 0.95]$ (см. документацию к классу).

При решении можно пользоваться `lightning` или писать цикл обучения вручную. В последнем случае не забудьте вручную отправить модель и батчи на GPU, чтобы обучалось быстрее

### Задание 1 (3 балла). Цикл обучения с расчётом Mean Average Precision

Запустите обучение модели из практики на всём обучающем датасете, выведите значение $mAP$ на валидационном датасете после окончания обучения.

В этом задании добейтесь $mAP > 0.3$, если всё сделано правильно - для этого должно хватать 30-50 эпох.

In [52]:
# ! wget https://tudatalib.ulb.tu-darmstadt.de/bitstream/handle/tudatalib/3799/yeast_cell_in_microstructures_dataset.zip
# ! unzip yeast_cell_in_microstructures_dataset.zip -d yeast_cell_in_microstructures_dataset

In [ ]:
# ! pip install --upgrade torch torchvision pytorch-lightning


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
# ! pip install albumentations


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
# ! pip install lightning


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [23]:
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.figure import Figure
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics.functional.detection import intersection_over_union
from torchvision.ops.boxes import box_convert

import lightning.pytorch as L
from lightning.pytorch.utilities.types import EVAL_DATALOADERS, TRAIN_DATALOADERS, STEP_OUTPUT
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from lightning.pytorch import Trainer
import torchmetrics
from torchmetrics.detection import MeanAveragePrecision
from torchvision.ops.boxes import nms
from typing import Any
import numpy as np
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR, ReduceLROnPlateau
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
from torch_ema import ExponentialMovingAverage

torch.manual_seed(42)
L.seed_everything(42)

torch.cuda.empty_cache()

Seed set to 42


In [24]:
def create_metrics(prefix: str) -> torchmetrics.MetricCollection:
    return torchmetrics.MetricCollection(
        [
            MeanAveragePrecision(iou_type="bbox", box_format="cxcywh", class_metrics=True),
        ],
        prefix=prefix,
    )

In [25]:
def process_yolo_preds(preds: Tensor, rescaled_anchors: Tensor) -> tuple[Tensor, Tensor, Tensor]:
    """
    Преобразование выходов модели в
    1. Логит наличия объекта (вероятность получается применением сигмоиды)
    2. Положение рамки относительно ячейки в формате cxcywh
    3. Логиты классов (вероятности получаются применением softmax)
    """
    rescaled_anchors = rescaled_anchors.view(1, len(rescaled_anchors), 1, 1, 2).to(preds.device)
    box_predictions = preds[..., 1:5].clone()

    box_predictions[..., 0:2] = torch.sigmoid(box_predictions[..., 0:2])
    box_predictions[..., 2:] = torch.exp(box_predictions[..., 2:]) * rescaled_anchors

    scores = preds[..., 0:1]
    return scores, box_predictions, preds[..., 5:]

In [32]:
GRID_SIZE = 8
IMAGE_SIZE = 256
ANCHORS = [
    [48, 72],
    #[64, 64],
    #[72, 48],
]

rescaled_anchors = torch.tensor(ANCHORS) / IMAGE_SIZE * GRID_SIZE
rescaled_anchors = rescaled_anchors.to('cuda')

In [33]:
def iou_wh(wh1: Tensor, wh2: Tensor) -> Tensor:
    intersection_area = torch.min(wh1[..., 0], wh2[..., 0]) * torch.min(wh1[..., 1], wh2[..., 1])

    box1_area = wh1[..., 0] * wh1[..., 1]
    box2_area = wh2[..., 0] * wh2[..., 1]
    union_area = box1_area + box2_area - intersection_area

    iou_score = intersection_area / union_area
    return iou_score


def boxes_to_cells(
    boxes: Tensor,
    classes: Tensor,
    rescaled_anchors: Tensor,
    grid_size: int = 8,
    ignore_iou_thresh: float = 0.5,
) -> Tensor:
    targets = torch.zeros((len(rescaled_anchors), grid_size, grid_size, 6))

    for box, class_label in zip(boxes, classes):
        iou_anchors = iou_wh(box[2:4], rescaled_anchors / grid_size)
        anchor_indices = iou_anchors.argsort(descending=True, dim=0)
        x, y, width, height = box

        has_anchor = False
        for anchor_idx in anchor_indices:
            s = grid_size

            i, j = int(s * y), int(s * x)
            anchor_taken = targets[anchor_idx, i, j, 0]

            if not anchor_taken and not has_anchor:
                x_cell, y_cell = s * x - j, s * y - i
                width_cell, height_cell = (width * s, height * s)
                box_coordinates = torch.tensor([x_cell, y_cell, width_cell, height_cell])

                targets[anchor_idx, i, j, 0] = 1
                targets[anchor_idx, i, j, 1:5] = box_coordinates
                targets[anchor_idx, i, j, 5] = int(class_label)

                has_anchor = True

            elif not anchor_taken and iou_anchors[anchor_idx] > ignore_iou_thresh:
                targets[anchor_idx, i, j, 0] = -1

    return targets

In [34]:
class YeastDetectionDataset(Dataset):
    def __init__(
        self, subset_dir: Path, anchors: list[tuple[int, int]], image_size: int, grid_size: int = 8
    ) -> None:
        super().__init__()
        self.subset_dir = subset_dir
        self.items = list((self.subset_dir / "inputs").glob("*.pt"))
        self.ignore_iou_thresh = 0.5
        self.rescaled_anchors = torch.tensor(anchors) / image_size * grid_size
        self.grid_size = grid_size
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor]:
        image_path = self.items[index]
        image = torch.load(image_path, weights_only=True).unsqueeze(0)
        classes = (
            torch.load(self.subset_dir / "classes" / image_path.parts[-1], weights_only=True) + 1
        )
        boxes = torch.load(
            self.subset_dir / "bounding_boxes" / image_path.parts[-1], weights_only=True
        )
        boxes = box_convert(boxes, "xyxy", "cxcywh") / self.image_size

        targets = boxes_to_cells(
            boxes, classes, self.rescaled_anchors, self.grid_size, self.ignore_iou_thresh
        )
        return image, targets

In [35]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.activation = nn.LeakyReLU(0.1)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return self.activation(x)

class TinyYOLO(nn.Module):
    def __init__(self, num_classes: int = 2, num_anchors: int = 1, in_channels: int = 1) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.num_anchors = num_anchors
        self.layers = nn.Sequential(
            CNNBlock(1, 16, kernel_size=3, stride=2, padding=1, dilation=2),
            CNNBlock(16, 32, kernel_size=3, stride=2, padding=1, dilation=2),
            CNNBlock(32, 64, kernel_size=3, stride=2, padding=1, dilation=2),
            CNNBlock(64, 128, kernel_size=3, stride=2, padding=1, groups=8),
            CNNBlock(128, 256, kernel_size=3, stride=1, padding=1, groups=8),
            CNNBlock(256, 256, kernel_size=3, stride=1, padding=1, groups=16),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256, num_anchors * (num_classes + 5), kernel_size=1)
        )
    
    def forward(self, x: Tensor) -> Tensor:
        x = self.layers(x)
        B, _, W, H = x.shape
        x = x.view(B, self.num_anchors, self.num_classes + 5, W, H)
        x = x.permute(0, 1, 3, 4, 2)
        return x


In [36]:
def cells_to_bboxes(cells: Tensor, rescaled_anchors: Tensor, is_predictions=False) -> Tensor:
    if is_predictions:
        scores, box_predictions, logits = process_yolo_preds(cells, rescaled_anchors)
        scores = torch.sigmoid(scores)
        best_class = torch.argmax(logits, dim=-1).unsqueeze(-1) + 1

    else:
        box_predictions = cells[..., 1:5].clone()
        scores = cells[..., 0:1]
        best_class = cells[..., 5:6]

    _, _, H, W, _ = cells.shape
    range_y, range_x = torch.meshgrid(
        torch.arange(H, dtype=cells.dtype, device=cells.device),
        torch.arange(W, dtype=cells.dtype, device=cells.device),
        indexing="ij",
    )
    x = torch.cat(
        [
            best_class,
            scores,
            (box_predictions[..., 0:1] + range_x[None, None, :, :, None]) / W,
            (box_predictions[..., 1:2] + range_y[None, None, :, :, None]) / H,
            box_predictions[..., 2:3] / W,
            box_predictions[..., 3:4] / H,
        ],
        -1,
    )

    return x.view(-1, 6)

In [37]:
class YOLOLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss()
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, pred: Tensor, target: Tensor, anchors: Tensor) -> Tensor:
        pred = pred.clone()
        target = target.clone()

        obj = target[..., 0] == 1
        no_obj = target[..., 0] == 0

        scores, pred_boxes, logits = process_yolo_preds(pred, anchors)

        no_object_loss = self.bce(
            (scores[no_obj]),
            (target[..., 0:1][no_obj]),
        )

        ious = intersection_over_union(pred_boxes[obj], target[..., 1:5][obj]).detach()
        object_loss = self.mse(scores[obj].sigmoid(), ious * target[..., 0:1][obj])

        anchors = anchors.reshape(1, len(anchors), 1, 1, 2)
        pred[..., 1:3] = pred[..., 1:3].sigmoid()
        target[..., 3:5] = torch.log(1e-6 + target[..., 3:5] / anchors)
        box_loss = self.mse(pred[..., 1:5][obj], target[..., 1:5][obj])

        class_loss = self.cross_entropy(logits[obj], target[..., 5][obj].long() - 1)

        return box_loss + object_loss + no_object_loss + class_loss

In [38]:
class YoloDataModule(L.LightningDataModule):
    def __init__(
        self,
        datadir: Path,
        batch_size: int,
        anchors: list,
        image_size: int = 256,
        grid_size: int = 8,
        train_subdir: str = "train",
        val_subdir: str = "val",
        test_subdir: str = "test",
        num_workers: int = 15,
    ) -> None:
        super().__init__()
        self.batch_size = batch_size
        self.anchors = anchors
        self.image_size = image_size
        self.grid_size = grid_size
        datadir = Path(datadir)
        self.train_dir = datadir / train_subdir
        self.val_dir = datadir / val_subdir
        self.test_dir = datadir / test_subdir
        self._train_dataset = None
        self._val_dataset = None
        self._test_dataset = None
        self.num_workers = num_workers

    def setup(self, stage: str) -> None:
        if stage in ("fit", "validate"):
            self.val_dataset = YeastDetectionDataset(self.val_dir, self.anchors, self.image_size, self.grid_size)
        if stage == "fit":
            self.train_dataset = YeastDetectionDataset(self.train_dir, self.anchors, self.image_size, self.grid_size)
        elif stage == "test":
            self.test_dataset = YeastDetectionDataset(self.test_dir, self.anchors, self.image_size, self.grid_size)

    def train_dataloader(self) -> TRAIN_DATALOADERS:
        if self._train_dataset is None:
            self.setup("fit")
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self) -> EVAL_DATALOADERS:
        if self._val_dataset is None:
            self.setup("validate")
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )

    def test_dataloader(self) -> EVAL_DATALOADERS:
        if self._test_dataset is None:
            self.setup("test")
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )

In [39]:
class YOLOLit(L.LightningModule):
    def __init__(self, model: nn.Module, anchors: Tensor, learning_rate: float) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=['model', 'anchors'])
        self.model = model
        self.anchors = anchors
        self.learning_rate = learning_rate

        self.func = YOLOLoss()

        self.map_metric = MeanAveragePrecision(iou_type="bbox", box_format="cxcywh", class_metrics=True)

    def training_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT:

        x, y = batch
        y_hat = self.model(x)
        loss = self.func(y_hat, y, self.anchors)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT | None:
        x, y = batch
        y_hat = self.model(x)
        loss = self.func(y_hat, y, self.anchors)
        
        predicted_boxes = []
        target_boxes = []
        for i in range(len(y)):
            pred_boxes = cells_to_bboxes(y_hat[i:i+1], self.anchors, is_predictions=True)

            pred_boxes = pred_boxes[nms(box_convert(pred_boxes[:, 2:], "cxcywh", "xyxy"), pred_boxes[:, 1], iou_threshold=0.3)]
            predicted_boxes.append(
                dict(
                    boxes=pred_boxes[:, 2:],
                    scores=pred_boxes[:, 1],
                    labels=pred_boxes[:, 0].long(),
                )
            )

            true_boxes = cells_to_bboxes(y[i:i+1], self.anchors)
            true_boxes = true_boxes[true_boxes[:, 1] == 1]

            target_boxes.append(
                dict(
                    boxes=true_boxes[:, 2:],
                    labels=true_boxes[:, 0].long(),
                )
            )

        self.map_metric.update(predicted_boxes, target_boxes)
        self.log("val_loss", loss, on_epoch=True, on_step=False, prog_bar=True)
        return loss
    
    def on_validation_epoch_end(self):
        map_metrics = self.map_metric.compute()
        
        self.log("val_map", map_metrics["map"], prog_bar=True)
        self.log("val_map_50", map_metrics["map_50"], prog_bar=True)

    def configure_optimizers(self) -> dict[str, Any]:
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)
        return {
            "optimizer": optimizer,
            "lr_scheduler": torch.optim.lr_scheduler.MultiStepLR(
                optimizer,
                milestones=[15, 20, 15],
                gamma=0.1,
            ),

        }

In [40]:
callbacks = [
    ModelCheckpoint(
        filename="{epoch}-{val_MeanAveragePrecision:.4f}",
        monitor="val_map",
        mode="max",
        save_top_k=2,
        save_last=True,
    ),
    EarlyStopping(
        monitor="val_map",
        mode="max",
        patience=5,
        min_delta=0.01
    ),
]

In [41]:
dataset_dir = Path("yeast_cell_in_microstructures_dataset")
train_dir = dataset_dir / "train"
datamodule = YoloDataModule(datadir=dataset_dir, batch_size=8, num_workers=15, anchors=ANCHORS)

logger = CSVLogger(save_dir="logs_task1", name="yolo")
model = TinyYOLO(in_channels=1, num_anchors=1, num_classes=2)

model_lit = YOLOLit(model=model, learning_rate=1e-3, anchors=rescaled_anchors)

trainer = Trainer(
    max_epochs=50,
    logger=logger,
    callbacks=callbacks,
    accelerator="auto",
    log_every_n_steps=3,
)

trainer.fit(model_lit, datamodule=datamodule)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type                 | Params | Mode 
------------------------------------------------------------
0 | model      | TinyYOLO             | 109 K  | train
1 | func       | YOLOLoss             | 0      | train
2 | map_metric | MeanAveragePrecision | 0      | train
------------------------------------------------------------
109 K     Trainable params
0         Non-trainable params
109 K     Total params
0.438     Total estimated model params size (MB)
33        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [42]:
val_results = trainer.validate(model_lit, datamodule=datamodule)
final_map = val_results[0]["val_map"]
print(f"Final map: {final_map:.4f}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        val_loss            0.0925239846110344
         val_map             0.427826851606369
       val_map_50           0.8692440986633301
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Final map: 0.4278


### Задание 2 (2 балла, необязательное). YOLOv3 loss
Мы упоминали, что на практике использовалась не совсем та же самая ошибка, что и в YOLOv3. В этом задании исправьте в классе YoloLoss ошибку регрессии, приведя её в соответствие с тем, как она описана в статье [YOLOv3: An Incremental Improvement](https://arxiv.org/abs/1804.02767) (см. раздел 2.1. Bounding Box Prediction).

Запустите обучение с изменённой ошибкой, добейтесь $mAP > 0.3$

Примечание: исправление само по себе очень небольшое, в 2 строчки кода.

### Задание 3 (3 балла). Выбор anchors с помощью кластеризации

В статье [YOLO9000: Better, Faster, Stronger](https://arxiv.org/abs/1612.08242) в разделе 2. Better. Dimension clusters описан способ выбора anchor boxes через кластеризацию обучающего датасета.

Проделайте то же самое с вашим обучающим датасетом, чтобы выбрать несколько anchor boxes.

В качестве результата выведите получившиеся размеры anchors для # Clusters = 5

In [70]:
def kmeans_iou_boxes(boxes, k=5):
    iters = 100
    boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
    n_boxes = boxes_tensor.shape[0]
    indices = torch.randperm(n_boxes)[:k]
    centroids = boxes_tensor[indices]
    
    for i in range(iters):
        boxes_expanded = boxes_tensor.unsqueeze(1)
        centroids_expanded = centroids.unsqueeze(0)

        boxes_broadcast = boxes_expanded.expand(-1, k, -1)
        centroids_broadcast = centroids_expanded.expand(n_boxes, -1, -1)

        ious = iou_wh(boxes_broadcast, centroids_broadcast)
        dists = (1 - ious) ** 2

        min_dists, cluster_ass = torch.min(dists, dim=1)

        new_centroids = []
        cluster_sizes = []
        
        for cluster_idx in range(k):
            mask = (cluster_ass == cluster_idx)
            cluster_boxes = boxes_tensor[mask]
            cluster_size = len(cluster_boxes)
            cluster_sizes.append(cluster_size)
            
            if cluster_size > 0:
                new_centroid = torch.mean(cluster_boxes, dim=0)
                new_centroids.append(new_centroid)
            else:
                new_centroids.append(centroids[cluster_idx])
        
        new_centroids = torch.stack(new_centroids)
        
        if torch.allclose(centroids, new_centroids, rtol=1e-4):
            break
            
        centroids = new_centroids
    
    avg_iou = 1 - torch.mean(dists).item()
    
    return centroids, avg_iou, cluster_sizes

In [71]:
def get_boxes_from_ds(dataset_dir, image_size=256):
    train_dir = Path(dataset_dir) / "train"
    inputs_dir = train_dir / "inputs"
    
    all_boxes = []
    
    for image_file in inputs_dir.glob("*.pt"):
        bbox_file = train_dir / "bounding_boxes" / image_file.name
        if bbox_file.exists():
            boxes = torch.load(bbox_file, weights_only=True)
            boxes_cxcywh = box_convert(boxes, "xyxy", "cxcywh")
            
            for box in boxes_cxcywh:
                cx, cy, w, h = box
                all_boxes.append([w, h])
    
    return all_boxes

In [72]:
boxes = get_boxes_from_ds("yeast_cell_in_microstructures_dataset")
centroids, avg_iou, cluster_sizes = kmeans_iou_boxes(boxes)
anchors_pixels = []
for i, centroid in enumerate(centroids):
    w, h = centroid.tolist()
    anchors_pixels.append([w, h])

print(torch.tensor(anchors_pixels))
print()

rescaled_anchors_new = torch.tensor(anchors_pixels) / IMAGE_SIZE * GRID_SIZE
rescaled_anchors_new = rescaled_anchors_new.to('cuda')
print(rescaled_anchors_new)

tensor([[ 43.4138,  45.2345],
        [ 57.5441,  87.9118],
        [ 71.9960, 104.9920],
        [ 55.5328,  62.3974],
        [ 76.5926,  68.4074]])

tensor([[1.3567, 1.4136],
        [1.7983, 2.7472],
        [2.2499, 3.2810],
        [1.7354, 1.9499],
        [2.3935, 2.1377]], device='cuda:0')


### Задание 4 (4 балла + бонусы за лучшую точность). Обучите модель


Ваша цель: $mAP > 0.6$ на валидации.

Можете использовать весь арсенал:
- использование множества якорных рамок (сформированных вручную или в результате кластеризации)
- любые изменения функции ошибки
- любые изменения архитектуры модели и регуляризация
- аугментации (вспоминаем `torchvision.transforms` и `albumentations`)
- любая длительность обучения, оптимизатор, расписание для learning rate

Бонусы за повышенную точность:
- **5 баллов**: $mAP > 0.65$
- **1 балл** за каждые следующие $0.01$ (т. е. за $mAP > 0.72$ в этом задании вы получите $4 + 12 = 16$ баллов)

**Важно**: перез запуском обучения зафиксируйте `torch.manual_seed()`

In [13]:
GRID_SIZE = 8
IMAGE_SIZE = 256
ANCHORS = [
    [48, 72],
    [64, 64],
    [72, 48],
]

rescaled_anchors = torch.tensor(ANCHORS) / IMAGE_SIZE * GRID_SIZE
rescaled_anchors = rescaled_anchors.to('cuda')

In [15]:
def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.3),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        
        A.GaussianBlur(blur_limit=3, p=0.2),
        A.GaussNoise(p=0.2),
        A.MotionBlur(blur_limit=5, p=0.2),

        A.Normalize(mean=[0.5], std=[0.5]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(
        format='yolo', 
        label_fields=['class_labels'],
        min_visibility=0.5,
        min_area=8
    ))

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=[0.5], std=[0.5]),
        ToTensorV2(),
    ])

class AlbumentationsYeastDataset(YeastDetectionDataset):
    def __init__(
        self, 
        subset_dir: Path, 
        anchors: list, 
        image_size: int = 256, 
        grid_size: int = 8, 
        is_train: bool = False
    ):
        super().__init__(subset_dir, anchors, image_size, grid_size)
        self.is_train = is_train
        
        if is_train:
            self.transform = get_train_transforms()
        else:
            self.transform = get_val_transforms()

    def __getitem__(self, index):
        image, targets = super().__getitem__(index)
        image_np = image.squeeze().numpy()
        
        if self.is_train:
            true_boxes = cells_to_bboxes(targets.unsqueeze(0), self.rescaled_anchors)
            true_boxes = true_boxes[true_boxes[:, 1] == 1]
            
            bboxes = []
            class_labels = []
            for box in true_boxes:
                class_id, _, cx, cy, w, h = box
                
                if w > 0 and h > 0 and cx >= 0 and cy >= 0 and cx <= 1 and cy <= 1:
                    bboxes.append([cx, cy, w, h])
                    class_labels.append(class_id.item() - 1)
            
            if bboxes:
                transformed = self.transform(
                    image=image_np,
                    bboxes=bboxes,
                    class_labels=class_labels
                )
                image_np = transformed['image']
                bboxes = transformed['bboxes']
                class_labels = transformed['class_labels']
            else:
                transformed = self.transform(image=image_np)
                image_np = transformed['image']
            
            if bboxes:
                boxes_tensor = torch.tensor(bboxes, dtype=torch.float32)
                classes_tensor = torch.tensor(class_labels, dtype=torch.long) + 1

                targets = boxes_to_cells(
                    boxes_tensor, classes_tensor, self.rescaled_anchors, 
                    self.grid_size, self.ignore_iou_thresh
                )
                
        else:
            transformed = self.transform(image=image_np)
            image_np = transformed['image']
        
        image = image_np
        return image, targets

class AlbumentationsDataModule(YoloDataModule):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        
    def setup(self, stage: str):
        if stage in ("fit", "validate"):
            self.val_dataset = AlbumentationsYeastDataset(
                self.val_dir, self.anchors, self.image_size, 
                self.grid_size, is_train=False
            )
        if stage == "fit":
            self.train_dataset = AlbumentationsYeastDataset(
                self.train_dir, self.anchors, self.image_size, 
                self.grid_size, is_train=True
            )

In [16]:
class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.layer = nn.Sequential(
            CNNBlock(channels, channels // 2, kernel_size=1, stride=1),
            CNNBlock(channels // 2, channels, kernel_size=3, stride=1, padding=1)
        )

    def forward(self, x: Tensor) -> Tensor:
        return x + self.layer(x)

In [17]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.activation = nn.LeakyReLU(0.1)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return self.activation(x)

class LargeYOLO(nn.Module):
    def __init__(self, num_classes: int = 2, num_anchors: int = 1, in_channels: int = 1) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.num_anchors = num_anchors
        
        self.layers = nn.Sequential(
            CNNBlock(in_channels, 32, kernel_size=3, stride=1, padding=1),
            CNNBlock(32, 64, kernel_size=3, stride=1, padding=1),
            
            CNNBlock(64, 128, kernel_size=3, stride=2, padding=1),
            ResidualBlock(128),
            
            CNNBlock(128, 256, kernel_size=3, stride=2, padding=1),
            ResidualBlock(256),
            ResidualBlock(256),
            
            CNNBlock(256, 512, kernel_size=3, stride=2, padding=1),
            ResidualBlock(512),
            ResidualBlock(512),
            
            CNNBlock(512, 1024, kernel_size=3, stride=2, padding=1),
            ResidualBlock(1024),
            
            CNNBlock(1024, 512, kernel_size=1, stride=1, padding=0),
            CNNBlock(512, 1024, kernel_size=3, stride=1, padding=1),
            CNNBlock(1024, 512, kernel_size=1, stride=1, padding=0),
            CNNBlock(512, 1024, kernel_size=3, stride=1, padding=1),
            CNNBlock(1024, 512, kernel_size=1, stride=1, padding=0),
            nn.Dropout2d(0.2),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(512, num_anchors * (num_classes + 5), kernel_size=1)
        )
    
    def forward(self, x: Tensor) -> Tensor:
        x = self.layers(x)
        B, _, W, H = x.shape
        x = x.view(B, self.num_anchors, self.num_classes + 5, W, H)
        x = x.permute(0, 1, 3, 4, 2)
        return x

In [18]:
class LargeYOLOLoss(nn.Module):
    def __init__(self, lambda_coord=5.0, lambda_noobj=0.6, label_smoothing=0.05, iou_weight=1.2):
        super().__init__()
        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss()
        self.cross_entropy = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        
        self.lambda_coord = lambda_coord
        self.lambda_noobj = lambda_noobj
        self.iou_weight = iou_weight

    def forward(self, pred: Tensor, target: Tensor, anchors: Tensor) -> Tensor:
        pred = pred.clone()
        target = target.clone()

        obj = target[..., 0] == 1
        no_obj = target[..., 0] == 0

        scores, pred_boxes, logits = process_yolo_preds(pred, anchors)

        no_object_loss = self.bce(scores[no_obj], target[..., 0:1][no_obj])
        object_loss = self.bce(scores[obj], target[..., 0:1][obj])

        anchors = anchors.reshape(1, len(anchors), 1, 1, 2)
        pred[..., 1:3] = pred[..., 1:3].sigmoid()
        target[..., 3:5] = torch.log(1e-6 + target[..., 3:5] / anchors)
        box_loss = self.mse(pred[..., 1:5][obj], target[..., 1:5][obj])

        class_loss = self.cross_entropy(logits[obj], target[..., 5][obj].long() - 1)

        iou_loss = torch.tensor(0.0, device="cuda")
        if obj.sum() > 0:
            ious = intersection_over_union(pred_boxes[obj], target[..., 1:5][obj])
            iou_loss = (1.0 - ious).mean()

        total_loss = (
            self.lambda_coord * box_loss + 
            object_loss + 
            self.lambda_noobj * no_object_loss + 
            class_loss +
            self.iou_weight * iou_loss
        )
        
        return total_loss

In [19]:
class LargeYOLOLit(L.LightningModule):
    def __init__(self, model: nn.Module, anchors: Tensor, learning_rate: float) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=['model', 'anchors'])
        self.model = model
        self.anchors = anchors
        self.learning_rate = learning_rate

        self.func = LargeYOLOLoss()

        self.map_metric = MeanAveragePrecision(iou_type="bbox", box_format="cxcywh", class_metrics=True)

    def training_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT:

        x, y = batch
        y_hat = self.model(x)
        loss = self.func(y_hat, y, self.anchors)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT | None:
        x, y = batch
        y_hat = self.model(x)
        loss = self.func(y_hat, y, self.anchors)
        
        predicted_boxes = []
        target_boxes = []
        for i in range(len(y)):
            pred_boxes = cells_to_bboxes(y_hat[i:i+1], self.anchors, is_predictions=True)

            pred_boxes = pred_boxes[nms(box_convert(pred_boxes[:, 2:], "cxcywh", "xyxy"), pred_boxes[:, 1], iou_threshold=0.3)]
            predicted_boxes.append(
                dict(
                    boxes=pred_boxes[:, 2:],
                    scores=pred_boxes[:, 1],
                    labels=pred_boxes[:, 0].long(),
                )
            )

            true_boxes = cells_to_bboxes(y[i:i+1], self.anchors)
            true_boxes = true_boxes[true_boxes[:, 1] == 1]

            target_boxes.append(
                dict(
                    boxes=true_boxes[:, 2:],
                    labels=true_boxes[:, 0].long(),
                )
            )

        self.map_metric.update(predicted_boxes, target_boxes)
        self.log("val_loss", loss, on_epoch=True, on_step=False, prog_bar=True)
        return loss
    
    def on_validation_epoch_end(self):
        map_metrics = self.map_metric.compute()
        
        self.log("val_map", map_metrics["map"], prog_bar=True)
        self.log("val_map_50", map_metrics["map_50"], prog_bar=True)

    def configure_optimizers(self) -> dict[str, Any]:
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.learning_rate, weight_decay=1e-3, betas=(0.9, 0.999))
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=self.learning_rate,
            epochs=self.trainer.max_epochs,
            steps_per_epoch=len(self.trainer.datamodule.train_dataloader()),
            pct_start=0.1,
            div_factor=10.0,
            final_div_factor=100.0
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": scheduler

        }

In [20]:
callbacks_large = [
    ModelCheckpoint(
        filename="{epoch}-{val_MeanAveragePrecision:.4f}",
        monitor="val_map",
        mode="max",
        save_top_k=2,
        save_last=True,
    ),
    EarlyStopping(
        monitor="val_map",
        mode="max",
        patience=10,
        min_delta=0.001
    ),
]

In [21]:
torch.manual_seed(42)

dataset_dir = Path("yeast_cell_in_microstructures_dataset")
train_dir = dataset_dir / "train"
datamodule_large = AlbumentationsDataModule(datadir=dataset_dir, batch_size=8, num_workers=15, anchors=ANCHORS)

logger_large = CSVLogger(save_dir="logs_task4", name="yolo")
model_large = LargeYOLO(in_channels=1, num_anchors=3, num_classes=2)

model_lit_large = LargeYOLOLit(model=model_large, learning_rate=1e-3, anchors=rescaled_anchors)

trainer_large = Trainer(
    max_epochs=500,
    logger=logger_large,
    callbacks=callbacks_large,
    accelerator="auto",
    log_every_n_steps=10
)

trainer_large.fit(model_lit_large, datamodule=datamodule_large)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/vladimir/dl_hw/.venv/lib/python3.13/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for pe

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [22]:
val_results_large = trainer_large.validate(model_lit_large, datamodule=datamodule_large)
final_map_large = val_results_large[0]["val_map"]
print(f"Final map: {final_map_large:.4f}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        val_loss            1.5770330429077148
         val_map            0.6730695366859436
       val_map_50           0.9570034146308899
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Final map: 0.6731
